# Fish Toxicty Regression

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit



df = pd.read_csv(
    "../adore_dataset/processed/s-F2F-1_mortality.csv",
    dtype=str
)

split_def = df["split_occurrence"]



starts_with_tax = [
    col for col in df.columns
    if col.startswith("tax")
]


starts_with_split = [
    col for col in df.columns
    if col.startswith("split")
]



cols_to_drop = [

    # other LC50 results
    # keep result_conc1_mean_binary for classification
    # keep result_conc1_mean_mol_log for regression
    "result_conc1_mean",
    "result_conc1_mean_mol",
    "result_conc1_mean_log",
    "result_conc1_mean_op",

    # IDs and names
    "test_id",
    "result_id",
    "reference_number",
    "test_cas",
    "test_cas_name",
    "chem_name",
    "chem_dtxsid",
    "chem_dtxcid",
    "chem_inchi",
    "chem_inchikey",
    "chem_pcp_cid",
    "chem_pcp_inchikey",
    "chem_pcp_inchi",
    "chem_pcp_iupac_name",
    "chem_pcp_can_smiles",
    "chem_rdkit_can_smiles",
    "chem_sf",

    # fingerprints
    "chem_pcp_fp",
    "chem_MACCS_fp",
    "chem_Morgan_fp",
    "chem_ToxPrint_fp",

    # not useful
    "test_location",
    "test_control_type",
    "test_application_freq_unit",
    "test_organism_lifestage",
    "result_effect",
    "result_endpoint",
    "result_conc1_type",
    "test_media_type",
    "chem_ws_binary",
    "chem_mol2vec_allowed",
    "media_ph_mean",
    "media_temperature_mean",
    "chem_pka_median",
    "species_number"
]


# add tax and split columns to columns to drop
cols_to_drop.extend(starts_with_tax)
cols_to_drop.extend(starts_with_split)


print("Original shape:", df.shape)

df = df.drop(
    columns=cols_to_drop,
    errors="ignore"
)

# add split_occurrence 
df["split_occurrence"] = split_def

print("Cleaned shape:", df.shape)


# define cols

numeric_cols = []

for col in df.columns:

    converted = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    if converted.notna().sum() == df[col].notna().sum():
        numeric_cols.append(col)


categorical_cols = [
    col for col in df.columns
    if col not in numeric_cols
]


Original shape: (4641, 1106)
Cleaned shape: (4641, 1034)


In [4]:
# test and train sets
test_set = df[
    df["split_occurrence"] == "test"
].copy()

train_set = df[
    df["split_occurrence"] != "test"
].copy()



# define split

test_fold = (
    train_set["split_occurrence"]
    .astype(int)
    .values
)

ps = PredefinedSplit(test_fold)

In [5]:


# define X and y for regression

X_train_reg = train_set.drop(
    columns=[
        "split_occurrence",
        "result_conc1_mean_binary",
        "result_conc1_mean_mol_log",
        "test_exposure_type"
    ]
)

y_train_reg = pd.to_numeric(
    train_set["result_conc1_mean_mol_log"],
    errors="coerce"
)


X_test_reg = test_set.drop(
    columns=[
        "split_occurrence",
        "result_conc1_mean_binary",
        "result_conc1_mean_mol_log",
        "test_exposure_type"
    ]
)

y_test_reg = pd.to_numeric(
    test_set["result_conc1_mean_mol_log"],
    errors="coerce"
)


print("Regression X train:", X_train_reg.shape)
print("Regression y train:", y_train_reg.shape)

print("Regression X test:", X_test_reg.shape)
print("Regression y test:", y_test_reg.shape)

Regression X train: (3928, 1030)
Regression y train: (3928,)
Regression X test: (713, 1030)
Regression y test: (713,)


In [7]:
# regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    ExtraTreesRegressor
)
from sklearn.neural_network import MLPRegressor

In [8]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42),
    "MLP": MLPRegressor(max_iter=2000, random_state=42)
}

In [ ]:
# cross validation
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    
    scores = cross_val_score(
        model,
        X_train_reg,
        y_train_reg,
        cv=ps,
        scoring="r2"
    )
    
    print(name, scores.mean())

Linear Regression 0.4150069340348299


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.14340697893312e-24.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Ridge -2.439102881862227


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.284376e+03, tolerance: 7.760e-01
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.382734e+03, tolerance: 7.963e-01
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or

Lasso 0.3632801990454838


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.211592e+03, tolerance: 7.760e-01
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.299423e+03, tolerance: 7.963e-01
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or

Elastic Net 0.3512807005277395
KNN 0.3379761348643117
SVR 0.043944614568197274
Decision Tree 0.17851906167686363
